In [5]:
###
# Imports
###
import cv2
from ultralytics import YOLO

###
# Step 1: Load pretrained model
###

# YOLOv8n = nano variant, smallest/fastest, good for CPU
# Auto-downloads weights on first run (~6MB)
model = YOLO("yolov8n.pt")

# YOLOv8 class index for bird
BIRD_CLASS_ID = 14
# print(model.names[BIRD_CLASS_ID]) 

###
# Step 2: Configure video capture
###
VIDEO_PATH = "4_data/birds.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise IOError(f"Cannot open video file: {VIDEO_PATH}")

###
# Step 3: Configure window for output
###
WINDOW_NAME = "Bird Detection (YOLOv8n)"
CONF_THRESHOLD = 0.3  # minimum confidence to draw a detection

###
# Step 4: Main loop - read, run inference, filter to birds, display
###
while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video or failed to grab frame")
        break

    frame = cv2.resize(frame, None, fx=0.5, fy=0.5)
    
    # Run inference on this frame
    results = model(frame, verbose=False)[0]

    # Filter detections to just the "bird" class above our confidence threshold
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        if cls_id == BIRD_CLASS_ID and conf >= CONF_THRESHOLD:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw label with confidence
            label = f"Bird {conf:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow(WINDOW_NAME, frame)

    # Exit on 'q' keypress
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

###
# Step 5: Cleanup
###
cap.release()
cv2.destroyAllWindows()